In [1]:
import numpy as np

import data.breathe_data as bd
import data.helpers as dh
import datetime
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

In [49]:
df_pre, start_idx, end_idx = dh.find_longest_conseq_sequence(
    df[df.ID == '523'], n_missing_days_allowed=1
)
df_pre[2:-1]

,ID,Date Recorded,FEV1,O2 Saturation,FEF2575,ecFEV1,ecFEF2575,Sex,Height,Age,...,ecFEV1 % Predicted,FEV1 % Predicted,O2 Saturation % Healthy,ecFEF2575%ecFEV1,idx ecFEV1 (L),idx O2 saturation (%),idx ecFEF2575%ecFEV1,idx ecFEF25-75 % ecFEV1 (%),Prev day,Days elapsed
61,523,2022-08-12,1.79,98,1.51,1.79,1.51,Female,166.0,22,...,51.589975,51.589975,99.901510,84.357542,35,48,42,42,2022-08-11,"1 day, 0:00:00"
62,523,2022-08-13,1.74,97,1.46,1.74,1.46,Female,166.0,22,...,50.148914,50.148914,98.882107,83.908046,34,47,41,41,2022-08-12,"1 day, 0:00:00"
63,523,2022-08-14,1.82,98,1.49,1.82,1.49,Female,166.0,22,...,52.454611,52.454611,99.901510,81.868132,36,48,40,40,2022-08-13,"1 day, 0:00:00"
64,523,2022-08-15,1.78,98,1.65,1.78,1.65,Female,166.0,22,...,51.301763,51.301763,99.901510,92.696629,35,48,46,46,2022-08-14,"1 day, 0:00:00"
65,523,2022-08-16,1.83,97,1.58,1.83,1.58,Female,166.0,22,...,52.742824,52.742824,98.882107,86.338798,36,47,43,43,2022-08-15,"1 day, 0:00:00"
66,523,2022-08-17,1.79,98,1.61,1.79,1.61,Female,166.0,22,...,51.589975,51.589975,99.901510,89.944134,35,48,44,44,2022-08-16,"1 day, 0:00:00"
67,523,2022-08-18,1.81,98,1.60,1.81,1.60,Female,166.0,22,...,52.166399,52.166399,99.901510,88.397790,36,48,44,44,2022-08-17,"1 day, 0:00:00"
68,523,2022-08-19,1.90,98,1.79,1.90,1.79,Female,166.0,22,...,54.760309,54.760309,99.901510,94.210526,38,48,47,47,2022-08-18,"1 day, 0:00:00"
69,523,2022-08-20,1.75,98,1.56,1.75,1.56,Female,166.0,22,...,50.437126,50.437126,99.901510,89.142857,35,48,44,44,2022-08-19,"1 day, 0:00:00"
70,523,2022-08-21,1.77,98,1.43,1.77,1.84,Female,166.0,22,...,51.013551,51.013551,99.901510,80.790960,35,48,40,40,2022-08-20,"1 day, 0:00:00"


In [65]:
df[
    (df.ID == "104")
    & (df["Date Recorded"] > datetime.date(2020, 5, 1))
    & (df["Date Recorded"] < datetime.date(2020, 11, 1))
]
# df_step_change = df.loc[2445:2475]
df_step_change = df.loc[2450:2464]

# Create a new date range with consecutive dates
start_date = df_step_change["Date Recorded"].min()
end_date = start_date + pd.Timedelta(days=len(df_step_change) - 1)
date_range = pd.date_range(start=start_date, end=end_date, freq="D")
df_step_change["Date Recorded"] = date_range
# df_step_change = df_step_change[df_step_change['ecFEV1'].diff() <= 0][1:-1]

df_constant = df.loc[1:20]

df_other = df.loc


fig = make_subplots(
    rows=3,
    cols=2,
    column_titles=["ecFEV1 (L)", "ecFEF2575%ecFEV1 (%)"],
    row_titles=[df_step_change.ID.unique()[0], df_constant.ID.unique()[0], df_pre[2:-1].ID.unique()[0]],
    shared_xaxes=True,
    x_title="Time (days)",
    horizontal_spacing=0.2,
)


def add_trace_for_df(df, fig, row, colour):
    fig.add_trace(
        go.Scatter(
            x=df.index - df.index[0],
            y=df["ecFEV1"],
            mode="lines+markers",
            name=f"ecFEV1 {df.ID.unique()[0]}",
        ),
        row=row,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df.index - df.index[0],
            y=df["ecFEF2575%ecFEV1"],
            mode="lines+markers",
            name=f"ecFEF2575%ecFEV1 {df.ID.unique()[0]}",
        ),
        row=row,
        col=2,
    )
    fig.data[-2].marker.color = colour
    fig.data[-1].marker.color = colour


add_trace_for_df(df_constant, fig, 1, "#0072b2")
add_trace_for_df(df_step_change, fig, 2, "#d55e00")
add_trace_for_df(df_pre[2:-1], fig, 3, "#009e73")

fig.update_yaxes(range=[1.2, 2.2], row=2, col=1)
fig.update_yaxes(range=[1, 2], row=1, col=1)
fig.update_yaxes(range=[40, 90], row=2, col=2)
fig.update_yaxes(range=[25, 65], row=1, col=2)

fig.update_layout(title="ecFEV1 Over Time", width=700, height=400, showlegend=False)
fig.update_xaxes(
    showgrid=False, type="category", tickmode="array", tickvals=[0, 10, 19], ticktext=["1", "10", "20"]
)
fig.update_yaxes(showgrid=False)


fig.show()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_50234/1287020728.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [14]:
df[df.ID == "104"]

,ID,Date Recorded,FEV1,O2 Saturation,FEF2575,ecFEV1,ecFEF2575,Sex,Height,Age,Predicted FEV1,Healthy O2 Saturation,ecFEV1 % Predicted,FEV1 % Predicted,O2 Saturation % Healthy,ecFEF2575%ecFEV1,idx ecFEV1 (L),idx O2 saturation (%),idx ecFEF2575%ecFEV1,idx ecFEF25-75 % ecFEV1 (%)
2318,104,2019-01-24,1.27,96,0.66,1.27,0.93,Female,143.0,25,2.50019,98.509166,50.796144,50.796144,97.452861,51.968504,25,46,25,25
2319,104,2019-02-08,1.32,96,0.80,1.32,0.80,Female,143.0,25,2.50019,98.509166,52.795992,52.795992,97.452861,60.606061,26,46,30,30
2320,104,2019-02-09,1.40,98,0.93,1.40,0.93,Female,143.0,25,2.50019,98.509166,55.995749,55.995749,99.483129,66.428571,28,48,33,33
2321,104,2019-02-11,1.47,95,1.05,1.47,1.05,Female,143.0,25,2.50019,98.509166,58.795537,58.795537,96.437727,71.428571,29,45,35,35
2322,104,2019-02-12,1.41,95,0.92,1.41,0.92,Female,143.0,25,2.50019,98.509166,56.395719,56.395719,96.437727,65.248227,28,45,32,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2535,104,2023-10-21,1.88,98,1.17,1.88,1.76,Female,143.0,25,2.50019,98.509166,75.194292,75.194292,99.483129,62.234043,37,48,31,31
2536,104,2023-10-25,1.70,96,1.09,2.74,1.09,Female,143.0,25,2.50019,98.509166,109.591681,67.994839,97.452861,64.117647,54,46,32,32
2537,104,2023-10-31,1.84,98,1.25,2.74,1.25,Female,143.0,25,2.50019,98.509166,109.591681,73.594414,99.483129,67.934783,54,48,33,33
2538,104,2023-11-06,1.92,99,1.23,1.92,1.23,Female,143.0,25,2.50019,98.509166,76.794171,76.794171,100.498263,64.062500,38,49,32,32


In [39]:
ids = [
    "132",
    "146",
    "177",
    "180",
    "202",
    "117",
    "131",
    "134",
    "191",
    "139",
    "253",
    "101",
    # Also from consec values
    "405",
    "272",
    "201",
    "203",
    "527",
    # For step change
    "104",
]

fig = make_subplots(
    rows=len(ids),
    cols=2,
    column_titles=["ecFEV1", "ecFEF2575%ecFEV1"],
    row_titles=ids,
    shared_xaxes=True,
    x_title="Time (days)",
)

for i, id in enumerate(ids):
    add_trace_for_df(df[df.ID == id].head(30), fig, i + 1)
    fig.update_yaxes(range=[0.5, 3.5], row=i + 1, col=1)
    fig.update_yaxes(range=[20, 150], row=i + 1, col=2)
fig.update_traces(marker=dict(size=4))

fig.update_layout(title="ecFEV1 Over Time", width=800, height=1600, showlegend=False)

fig.show()